
Step 1: Install Dependencies

In [ ]:
!pip install -q langchain langchain-community langchain-groq langchain-text-splitters
!pip install -q faiss-cpu chromadb
!pip install -q sentence-transformers
!pip install -q PyPDF2 pypdf
!pip install -q groq tiktoken
!pip install -q rouge-score nltk evaluate pandas
print('✅ All packages installed!')

✅ All packages installed!


Step 2: API Key Setup


In [ ]:
import os
from google.colab import userdata

try:
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    print('✅ API key loaded from Colab Secrets')
except:
    GROQ_API_KEY = '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'
    print('⚠️  Using hardcoded API key - use Colab Secrets for better security')

os.environ['GROQ_API_KEY'] = GROQ_API_KEY

⚠️  Using hardcoded API key - use Colab Secrets for better security


Step 3: Legal Knowledge Base — Document Ingestion

In [ ]:

LEGAL_DOCUMENTS = [
    {
        'title': 'Indian Contract Act, 1872 — Formation of Contract',
        'content': """
        The Indian Contract Act, 1872 governs contract law in India. A contract is an agreement
        enforceable by law. Every promise and every set of promises forming the consideration for
        each other is an agreement. An agreement is a contract when it is made by the free consent
        of parties competent to contract, for a lawful consideration and with a lawful object, and
        is not hereby expressly declared to be void.

        Essential elements of a valid contract:
        1. Offer and Acceptance: There must be a lawful offer by one party and lawful acceptance
           by the other party. The offer must be definite and communicated.
        2. Consideration: Something of value must be exchanged between parties. Consideration
           must be real and lawful. Past consideration is not valid in Indian law.
        3. Competency of Parties: Parties must be of sound mind, major (18 years), and not
           disqualified by law. A minor's contract is void ab initio.
        4. Free Consent: Consent must be free from coercion, undue influence, fraud,
           misrepresentation, and mistake.
        5. Lawful Object and Consideration: The object of the contract must not be illegal,
           immoral, or opposed to public policy.
        6. Not Expressly Declared Void: The contract must not be declared void under any law.

        Void agreements include: agreements made by incompetent parties, agreements under mutual
        mistake of fact, agreements with unlawful consideration, agreements restraining marriage,
        agreements restraining trade, and agreements restraining legal proceedings.
        """
    },
    {
        'title': 'Indian Contract Act — Breach and Remedies',
        'content': """
        Breach of Contract occurs when a party refuses or fails to perform their contractual
        obligations without lawful excuse. Breach can be actual (failure at the time of performance)
        or anticipatory (expressed intention not to perform before due date).

        Remedies for Breach of Contract:
        1. Damages: The primary remedy. Types include ordinary/compensatory damages (Hadley v
           Baxendale rule — damages must be foreseeable), special damages (specifically
           communicated consequences), nominal damages, exemplary/punitive damages.
        2. Specific Performance: Court orders party to perform the contract. Available when
           damages are inadequate remedy — typically for unique goods, land, or rare items.
        3. Injunction: Court order restraining breach. Available for negative covenants.
        4. Quantum Meruit: Recovery for partial performance — 'as much as deserved'.
        5. Rescission: Setting aside the contract and restoring parties to original positions.

        Doctrine of Frustration (Section 56): A contract becomes void if, after formation,
        performance becomes impossible or unlawful due to an unforeseen event. Examples:
        destruction of subject matter, death or incapacity of party in personal contracts,
        supervening illegality, outbreak of war.
        """
    },
    {
        'title': 'Consumer Protection Act, 2019 — Rights and Complaints',
        'content': """
        The Consumer Protection Act, 2019 replaced the 1986 Act and provides a modernized
        framework protecting consumer rights in India including e-commerce transactions.

        Consumer Rights under the Act:
        1. Right to Safety: Protection against goods/services hazardous to life and property.
        2. Right to Information: Right to be informed about quality, quantity, potency, purity,
           standard, and price of goods or services.
        3. Right to Choose: Access to variety of goods and services at competitive prices.
        4. Right to be Heard: Consumer interests will receive due consideration at appropriate forums.
        5. Right to Redressal: Right to seek redressal against unfair trade practices.
        6. Right to Consumer Education: Right to acquire knowledge and skill to be an informed consumer.

        Complaint Process and Jurisdiction:
        - District Commission: Claims up to Rs. 1 crore
        - State Commission: Claims between Rs. 1 crore to Rs. 10 crores
        - National Commission: Claims above Rs. 10 crores

        Unfair Trade Practices include: false representation, misleading advertisements,
        offering gifts/prizes with intent to deceive, hoarding, and black-marketing.
        Time limit for filing complaint: 2 years from date of cause of action.
        Central Consumer Protection Authority (CCPA) established to regulate matters.
        """
    },
    {
        'title': 'Information Technology Act, 2000 — Cyber Law Basics',
        'content': """
        The Information Technology Act, 2000 (IT Act) provides legal framework for electronic
        governance and addresses cybercrime in India. Amended significantly in 2008.

        Key Provisions:
        1. Legal Recognition of Electronic Documents: Electronic records and digital signatures
           have legal validity. Section 4 gives legal recognition to electronic records.
        2. Digital Signatures: Authenticate identity of sender. Valid under law via certifying
           authorities licensed by Controller of Certifying Authorities.
        3. Cyber Offences and Penalties:
           - Section 43: Unauthorized access, downloading, virus introduction — compensation
           - Section 66: Computer-related offences — imprisonment up to 3 years
           - Section 66A (struck down): Offensive messages (declared unconstitutional in Shreya Singhal v UOI)
           - Section 66B: Receiving stolen computer resources — imprisonment up to 3 years
           - Section 66C: Identity theft — imprisonment up to 3 years, fine up to Rs. 1 lakh
           - Section 66D: Cheating by personation — imprisonment up to 3 years
           - Section 67: Publishing obscene material — imprisonment up to 5 years
           - Section 69: Government power to intercept, monitor, decrypt information
        4. Intermediary Guidelines: Platforms must observe due diligence and cannot be liable
           for third-party content if they follow safe harbor provisions.
        5. Data Protection: Section 43A — reasonable security practices for sensitive personal data.
        """
    },
    {
        'title': 'Indian Penal Code — General Principles of Criminal Liability',
        'content': """
        The Indian Penal Code (IPC), 1860, now replaced by Bharatiya Nyaya Sanhita 2023,
        defines offences and prescribes punishments. Key principles:

        Elements of Crime:
        1. Actus Reus: The guilty act — physical component of the crime. Must be voluntary.
        2. Mens Rea: The guilty mind — mental element. Includes intention, knowledge, recklessness.
        3. Causation: The act must cause the prohibited consequence.
        4. Concurrence: Actus Reus and Mens Rea must coincide.

        General Defences/Exceptions:
        - Mistake of Fact (Section 76, 79): Bonafide mistake of fact is a valid defence.
        - Act of Judge (Section 77): Acts of judges done in good faith are protected.
        - Accident (Section 80): Acts done accidentally without criminal intent are excused.
        - Necessity (Section 81): Acts causing harm to prevent greater harm.
        - Infancy (Section 82, 83): Children below 7 are exempt; 7-12 years with maturity assessed.
        - Insanity (Section 84): Person of unsound mind cannot form mens rea.
        - Intoxication (Section 85, 86): Involuntary intoxication is a valid defence.
        - Consent (Section 87): Acts done with consent where no grievous hurt is intended.
        - Right of Private Defence (Section 96-106): Right to defend person and property.

        Punishments under IPC/BNS: Death, Imprisonment for life, Imprisonment (rigorous/simple),
        Forfeiture of property, Fine, and Community Service (new addition in BNS 2023).
        """
    },
    {
        'title': 'Constitution of India — Fundamental Rights',
        'content': """
        Part III of the Constitution of India (Articles 12-35) guarantees Fundamental Rights
        to all citizens. These rights are justiciable — enforceable by courts.

        Six Categories of Fundamental Rights:
        1. Right to Equality (Articles 14-18):
           - Article 14: Equality before law and equal protection of law
           - Article 15: Prohibition of discrimination on grounds of religion, race, caste, sex, place of birth
           - Article 16: Equality of opportunity in public employment
           - Article 17: Abolition of untouchability
           - Article 18: Abolition of titles

        2. Right to Freedom (Articles 19-22):
           - Article 19: Six freedoms including speech, assembly, movement, profession
           - Article 20: Protection in respect of conviction for offences
           - Article 21: Protection of life and personal liberty
           - Article 21A: Right to education (added by 86th Amendment)
           - Article 22: Protection against arbitrary arrest and detention

        3. Right against Exploitation (Articles 23-24):
           - Prohibition of traffic in human beings and forced labour
           - Prohibition of employment of children below 14 in hazardous factories

        4. Right to Freedom of Religion (Articles 25-28)
        5. Cultural and Educational Rights (Articles 29-30)
        6. Right to Constitutional Remedies (Article 32) — Dr. Ambedkar called this the
           'heart and soul' of the Constitution. Five writs: Habeas Corpus, Mandamus,
           Prohibition, Certiorari, Quo Warranto.

        Reasonable Restrictions: Rights are not absolute. Parliament can impose reasonable
        restrictions on grounds of sovereignty, security, public order, decency, and morality.
        """
    },
    {
        'title': 'Transfer of Property Act, 1882 — Sale of Immovable Property',
        'content': """
        The Transfer of Property Act, 1882 governs transfer of property inter vivos (between
        living persons) in India.

        Sale of Immovable Property (Sections 54-57):
        Definition: Sale is a transfer of ownership in exchange for a price paid, promised,
        part-paid, or part-promised.

        Essentials of Valid Sale:
        1. Competent parties (seller must be owner or authorized)
        2. Subject matter must be transferable property
        3. Price must be paid or promised
        4. Conveyance — transfer of title

        Registration Requirements:
        - Property worth Rs. 100 or more: Must be registered under Registration Act
        - Delivered by registered instrument signed by seller and attested by two witnesses

        Rights and Liabilities of Buyer and Seller:
        Seller's duties: Disclose defects, produce title deeds, answer questions about title,
        execute proper conveyance.
        Buyer's duties: Disclose facts increasing property value, pay purchase price, bear loss
        after sale deed is executed.

        Doctrine of Lis Pendens (Section 52): Transfer of property during pending litigation
        affecting the property is subject to the outcome of the litigation.

        Doctrine of Part Performance (Section 53A): Protects transferee who has taken
        possession pursuant to an unregistered agreement to sell.
        """
    },
    {
        'title': 'Companies Act, 2013 — Corporate Law Fundamentals',
        'content': """
        The Companies Act, 2013 regulates incorporation, responsibilities, and winding up of
        companies in India. Replaced the Companies Act, 1956.

        Types of Companies:
        1. Private Limited Company: Restricts right to transfer shares, limits members to 200,
           prohibits public invitation for subscription.
        2. Public Limited Company: No restriction on transfer, minimum 7 members, can invite public.
        3. One Person Company (OPC): Single member company, introduced by 2013 Act.
        4. Nidhi Company: For mutual benefit of members, primarily for savings.
        5. Section 8 Company: Non-profit objectives — charitable, scientific, artistic purposes.

        Director Responsibilities:
        - Duty of care, skill, and diligence
        - Fiduciary duty to act in company's best interest
        - Avoid conflicts of interest
        - Not to accept benefits from third parties
        - Disclose interest in any transaction

        Corporate Governance:
        - Board meetings: Minimum 4 per year, not more than 120 days gap between two meetings
        - Annual General Meeting: Once a year within 6 months of financial year end
        - Statutory Auditor: Independent auditor mandatory; rotation every 5 years for listed companies
        - Corporate Social Responsibility: Companies with net worth Rs. 500 crore+ or turnover
          Rs. 1000 crore+ or net profit Rs. 5 crore+ must spend 2% of average net profits on CSR.

        NCLT (National Company Law Tribunal): Quasi-judicial body resolving company disputes,
        merger approvals, and insolvency matters.
        """
    }
]

print(f'✅ Loaded {len(LEGAL_DOCUMENTS)} legal documents')
for doc in LEGAL_DOCUMENTS:
    word_count = len(doc['content'].split())
    print(f"  📄 {doc['title'][:60]} — {word_count} words")

✅ Loaded 8 legal documents
  📄 Indian Contract Act, 1872 — Formation of Contract — 231 words
  📄 Indian Contract Act — Breach and Remedies — 169 words
  📄 Consumer Protection Act, 2019 — Rights and Complaints — 187 words
  📄 Information Technology Act, 2000 — Cyber Law Basics — 203 words
  📄 Indian Penal Code — General Principles of Criminal Liability — 210 words
  📄 Constitution of India — Fundamental Rights — 227 words
  📄 Transfer of Property Act, 1882 — Sale of Immovable Property — 181 words
  📄 Companies Act, 2013 — Corporate Law Fundamentals — 216 words


Step 4: Text Chunking — Multiple Strategies

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter
from langchain_core.documents import Document

def create_chunks(documents, chunk_size=500, chunk_overlap=50, strategy='recursive'):
    """
    Create chunks from documents using different strategies.

    Args:
        documents: List of dicts with 'title' and 'content'
        chunk_size: Size of each chunk in characters
        chunk_overlap: Overlap between chunks
        strategy: 'recursive' or 'character'
    """
    if strategy == 'recursive':
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=['\n\n', '\n', '. ', ' ', ''],
            length_function=len
        )
    else:
        splitter = CharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separator='\n'
        )

    all_chunks = []
    for doc in documents:
        chunks = splitter.split_text(doc['content'])
        for i, chunk in enumerate(chunks):
            all_chunks.append(Document(
                page_content=chunk,
                metadata={
                    'source': doc['title'],
                    'chunk_id': i,
                    'chunk_size': chunk_size,
                    'strategy': strategy
                }
            ))
    return all_chunks

# Experiment with different chunk sizes
CHUNK_CONFIGS = [
    {'chunk_size': 256,  'chunk_overlap': 30,  'label': 'Small (256)'},
    {'chunk_size': 512,  'chunk_overlap': 50,  'label': 'Medium (512)'},
    {'chunk_size': 1024, 'chunk_overlap': 100, 'label': 'Large (1024)'},
]

print('📊 Chunking Analysis:')
print(f'{"Config":<20} {"Chunks":<10} {"Avg Length":<15}')
print('-' * 45)

for config in CHUNK_CONFIGS:
    chunks = create_chunks(LEGAL_DOCUMENTS, config['chunk_size'], config['chunk_overlap'])
    avg_len = sum(len(c.page_content) for c in chunks) / len(chunks)
    print(f"{config['label']:<20} {len(chunks):<10} {avg_len:<15.1f}")

# Use medium chunks for the main system
CHUNKS = create_chunks(LEGAL_DOCUMENTS, chunk_size=512, chunk_overlap=50)
print(f'\n✅ Using 512-char chunks: {len(CHUNKS)} total chunks')

📊 Chunking Analysis:
Config               Chunks     Avg Length     
---------------------------------------------
Small (256)          68         172.0          
Medium (512)         38         317.1          
Large (1024)         20         613.2          

✅ Using 512-char chunks: 38 total chunks


Embedding Models — Two Different Models

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import time

# Embedding Model 1: all-MiniLM-L6-v2
print('Loading Embedding Model 1: all-MiniLM-L6-v2 (384-dim)...')
t1 = time.time()
model_minilm = SentenceTransformer('all-MiniLM-L6-v2')
t1 = time.time() - t1
print(f'  ✅ Loaded in {t1:.1f}s')

# Embedding Model 2: all-mpnet-base-v2
print('Loading Embedding Model 2: all-mpnet-base-v2 (768-dim)...')
t2 = time.time()
model_mpnet = SentenceTransformer('all-mpnet-base-v2')
t2 = time.time() - t2
print(f'  ✅ Loaded in {t2:.1f}s')

def get_embeddings(texts, model, batch_size=32):
    """Generate embeddings for a list of texts."""
    return model.encode(texts, batch_size=batch_size, show_progress_bar=True,
                        normalize_embeddings=True)

# Test both models
test_text = ['What are the essential elements of a valid contract?']
emb1 = get_embeddings(test_text, model_minilm)
emb2 = get_embeddings(test_text, model_mpnet)
print(f'\n📐 Embedding Dimensions:')
print(f'  Model 1 (MiniLM):  {emb1.shape[-1]} dimensions')
print(f'  Model 2 (MPNet):   {emb2.shape[-1]} dimensions')

Loading Embedding Model 1: all-MiniLM-L6-v2 (384-dim)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  ✅ Loaded in 7.5s


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


📐 Embedding Dimensions:
  Model 1 (MiniLM):  384 dimensions
  Model 2 (MPNet):   768 dimensions


Vector Databases — FAISS and ChromaDB

In [ ]:
import faiss
import chromadb
import numpy as np

# Generate embeddings for all chunks
chunk_texts = [c.page_content for c in CHUNKS]
chunk_metadata = [c.metadata for c in CHUNKS]

print('Generating embeddings with Model 1 (MiniLM)...')
embeddings_minilm = get_embeddings(chunk_texts, model_minilm)

print('Generating embeddings with Model 2 (MPNet)...')
embeddings_mpnet = get_embeddings(chunk_texts, model_mpnet)

print(f'✅ Embeddings ready: {len(chunk_texts)} chunks')

# Vector DB 1: FAISS (Facebook AI Similarity Search)
class FAISSVectorStore:
    def __init__(self, embeddings, texts, metadata, dim):
        self.texts = texts
        self.metadata = metadata
        self.index = faiss.IndexFlatIP(dim)  # Inner product (cosine with normalized vectors)
        self.index.add(embeddings.astype(np.float32))
        print(f'  FAISS index created: {self.index.ntotal} vectors, dim={dim}')

    def search(self, query_embedding, k=3):
        query = query_embedding.reshape(1, -1).astype(np.float32)
        scores, indices = self.index.search(query, k)
        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx != -1:
                results.append({
                    'text': self.texts[idx],
                    'metadata': self.metadata[idx],
                    'score': float(score)
                })
        return results

print('\n🔵 Creating FAISS Index...')
faiss_store_minilm = FAISSVectorStore(embeddings_minilm, chunk_texts, chunk_metadata, 384)
faiss_store_mpnet  = FAISSVectorStore(embeddings_mpnet,  chunk_texts, chunk_metadata, 768)

# Vector DB 2: ChromaDB
print('\n🟢 Creating ChromaDB Collections...')
chroma_client = chromadb.Client()

def create_chroma_collection(name, embeddings, texts, metadata):
    try:
        chroma_client.delete_collection(name)
    except:
        pass
    collection = chroma_client.create_collection(name)
    ids = [f'doc_{i}' for i in range(len(texts))]
    # ChromaDB needs string metadata values
    str_meta = [{k: str(v) for k, v in m.items()} for m in metadata]
    collection.add(
        embeddings=embeddings.tolist(),
        documents=texts,
        metadatas=str_meta,
        ids=ids
    )
    print(f'  ChromaDB collection "{name}": {collection.count()} documents')
    return collection

chroma_minilm = create_chroma_collection('legal_minilm', embeddings_minilm, chunk_texts, chunk_metadata)
chroma_mpnet  = create_chroma_collection('legal_mpnet',  embeddings_mpnet,  chunk_texts, chunk_metadata)

print('\n✅ Both vector databases ready!')

Generating embeddings with Model 1 (MiniLM)...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Generating embeddings with Model 2 (MPNet)...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Embeddings ready: 38 chunks

🔵 Creating FAISS Index...
  FAISS index created: 38 vectors, dim=384
  FAISS index created: 38 vectors, dim=768

🟢 Creating ChromaDB Collections...
  ChromaDB collection "legal_minilm": 38 documents
  ChromaDB collection "legal_mpnet": 38 documents

✅ Both vector databases ready!


Step 7: Retrieval Functions

In [ ]:
def retrieve_from_faiss(query, model, faiss_store, k=3):
    """Retrieve top-k chunks from FAISS."""
    query_emb = model.encode([query], normalize_embeddings=True)
    return faiss_store.search(query_emb[0], k=k)

def retrieve_from_chroma(query, model, chroma_collection, k=3):
    """Retrieve top-k chunks from ChromaDB."""
    query_emb = model.encode([query], normalize_embeddings=True)
    results = chroma_collection.query(
        query_embeddings=query_emb.tolist(),
        n_results=k
    )
    output = []
    for i in range(len(results['documents'][0])):
        output.append({
            'text': results['documents'][0][i],
            'metadata': results['metadatas'][0][i],
            'score': 1 - results['distances'][0][i] if results['distances'] else 0
        })
    return output

# Quick test
test_q = 'What are the rights of consumers in India?'
faiss_results = retrieve_from_faiss(test_q, model_minilm, faiss_store_minilm)
print(f'🔍 Test Query: "{test_q}"')
print(f'\nFAISS Top Result:')
print(f'  Source: {faiss_results[0]["metadata"]["source"][:60]}')
print(f'  Score: {faiss_results[0]["score"]:.4f}')
print(f'  Text: {faiss_results[0]["text"][:200]}...')

🔍 Test Query: "What are the rights of consumers in India?"

FAISS Top Result:
  Source: Consumer Protection Act, 2019 — Rights and Complaints
  Score: 0.6822
  Text: Consumer Rights under the Act:
        1. Right to Safety: Protection against goods/services hazardous to life and property.
        2. Right to Information: Right to be informed about quality, quanti...


LLM Integration via Groq API


In [ ]:
from groq import Groq

groq_client = Groq(api_key=GROQ_API_KEY)

# Available free models on Groq
LLM_MODELS = {
    'llama3': 'llama-3.1-8b-instant',
    'gemma2': 'gemma2-9b-it',  # decommissioned
    'mixtral': 'mixtral-8x7b-32768'  # decommissioned
}

def build_prompt(query, retrieved_chunks):
    """Build a RAG prompt with retrieved legal context."""
    context_parts = []
    for i, chunk in enumerate(retrieved_chunks, 1):
        source = chunk['metadata'].get('source', 'Unknown')
        context_parts.append(f'[Source {i}: {source}]\n{chunk["text"]}')

    context = '\n\n'.join(context_parts)

    prompt = f"""You are an expert Indian legal assistant. Answer the user's legal question
based ONLY on the provided legal context. Be precise, cite the relevant law/section,
and structure your answer clearly.

--- LEGAL CONTEXT ---
{context}
--- END CONTEXT ---

User Question: {query}

Provide a clear, structured answer citing the relevant laws and sections from the context above.
If the context doesn't contain enough information, say so clearly."""

    return prompt

def query_llm(prompt, model_key='llama3', max_tokens=600):
    """Query a Groq-hosted LLM."""
    model_name = LLM_MODELS[model_key]
    response = groq_client.chat.completions.create(
        model=model_name,
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=max_tokens,
        temperature=0.2  # Low temperature for factual legal answers
    )
    return response.choices[0].message.content

print('✅ LLM functions ready. Available models:', list(LLM_MODELS.keys()))

✅ LLM functions ready. Available models: ['llama3', 'gemma2', 'mixtral']


Complete RAG Pipeline — System Configurations

In [ ]:
import time

def rag_pipeline(query, config):
    """
    Complete RAG pipeline.
    Config options:
      embedding_model: 'minilm' or 'mpnet'
      vector_db: 'faiss' or 'chroma'
      llm: 'llama3', 'gemma2', 'mixtral'
      top_k: number of chunks to retrieve
    """
    start = time.time()

    # Select embedding model
    emb_model = model_minilm if config['embedding_model'] == 'minilm' else model_mpnet

    # Retrieve relevant chunks
    if config['vector_db'] == 'faiss':
        store = faiss_store_minilm if config['embedding_model'] == 'minilm' else faiss_store_mpnet
        chunks = retrieve_from_faiss(query, emb_model, store, k=config['top_k'])
    else:
        collection = chroma_minilm if config['embedding_model'] == 'minilm' else chroma_mpnet
        chunks = retrieve_from_chroma(query, emb_model, collection, k=config['top_k'])

    # Build prompt and query LLM
    prompt = build_prompt(query, chunks)
    answer = query_llm(prompt, model_key=config['llm'])

    elapsed = time.time() - start

    return {
        'query': query,
        'answer': answer,
        'retrieved_chunks': chunks,
        'config': config,
        'latency': elapsed
    }

# Define two system configurations for comparison
CONFIG_A = {
    'name': 'Config A: MiniLM + FAISS + LLaMA-3',
    'embedding_model': 'minilm',
    'vector_db': 'faiss',
    'llm': 'llama3',
    'top_k': 3
}

CONFIG_B = {
    'name': 'Config B: MPNet + ChromaDB + LLaMA-3.3-70B',
    'embedding_model': 'mpnet',
    'vector_db': 'chroma',
    'llm': 'gemma2',   # key stays same, now points to llama-3.3-70b
    'top_k': 4
}

print('✅ RAG Pipeline ready with 2 configurations:')
print(f'  📌 {CONFIG_A["name"]}')
print(f'  📌 {CONFIG_B["name"]}')

✅ RAG Pipeline ready with 2 configurations:
  📌 Config A: MiniLM + FAISS + LLaMA-3
  📌 Config B: MPNet + ChromaDB + LLaMA-3.3-70B


Step 10: Demo — Ask Legal Questions!

In [ ]:
# Force update LLM models and Config B
LLM_MODELS['gemma2'] = 'llama-3.3-70b-versatile'
LLM_MODELS['mixtral'] = 'llama3-8b-8192'
CONFIG_B['llm'] = 'gemma2'
print('Updated:', LLM_MODELS)

Updated: {'llama3': 'llama-3.1-8b-instant', 'gemma2': 'llama-3.3-70b-versatile', 'mixtral': 'llama3-8b-8192'}


In [ ]:

# Test with legal questions

test_queries = [
    'What are the essential elements required for a valid contract under Indian law?',
    'What are the consumer rights and remedies available in India?',
    'What cyber crimes are defined under the IT Act and what are the penalties?',
    'What is the doctrine of frustration in contract law?',
    'Explain the fundamental rights guaranteed by the Indian Constitution.',
]

print('='*70)
print('DEMO: Legal Assistant RAG System')
print('='*70)

query = test_queries[0]  # Change index to try different questions

print(f'\n❓ Question: {query}\n')
print('-'*70)

# Run with Config A
print(f'\n🔵 {CONFIG_A["name"]}')
result_a = rag_pipeline(query, CONFIG_A)
print(f'   Latency: {result_a["latency"]:.2f}s')
print(f'   Retrieved {len(result_a["retrieved_chunks"])} chunks')
print(f'\n📝 Answer (Config A):')
print(result_a['answer'])

print('\n' + '-'*70)

# Run with Config B
print(f'\n🟢 {CONFIG_B["name"]}')
result_b = rag_pipeline(query, CONFIG_B)
print(f'   Latency: {result_b["latency"]:.2f}s')
print(f'   Retrieved {len(result_b["retrieved_chunks"])} chunks')
print(f'\n📝 Answer (Config B):')
print(result_b['answer'])

DEMO: Legal Assistant RAG System

❓ Question: What are the essential elements required for a valid contract under Indian law?

----------------------------------------------------------------------

🔵 Config A: MiniLM + FAISS + LLaMA-3
   Latency: 0.54s
   Retrieved 3 chunks

📝 Answer (Config A):
**Essential Elements of a Valid Contract under Indian Law**

A valid contract under Indian law requires the following essential elements:

1. **Offer and Acceptance**: There must be a lawful offer by one party and lawful acceptance by the other party. The offer must be definite and communicated. (Source 1: Indian Contract Act, 1872 — Formation of Contract)

2. **Consideration**: Something of value must be exchanged between parties. Consideration must be real and lawful. Past consideration is not valid in Indian law. (Source 1: Indian Contract Act, 1872 — Formation of Contract)

3. **Competency of Parties**: Parties must be of sound mind, major (18 years), and not disqualified from contracting.

Step 11: Comparative Evaluation

In [ ]:
import pandas as pd
import numpy as np

# Evaluate both configurations on all test queries

print('Running comparative evaluation on all test queries...')
print('(This may take ~1-2 minutes due to API calls)\n')

results_a = []
results_b = []

for i, query in enumerate(test_queries):
    print(f'  Query {i+1}/{len(test_queries)}: {query[:50]}...')

    try:
        ra = rag_pipeline(query, CONFIG_A)
        results_a.append(ra)
        time.sleep(0.5)  # Rate limiting

        rb = rag_pipeline(query, CONFIG_B)
        results_b.append(rb)
        time.sleep(0.5)
    except Exception as e:
        print(f'    ⚠️ Error: {e}')


# Compute Metrics
def avg_retrieval_score(results):
    scores = []
    for r in results:
        chunk_scores = [c['score'] for c in r['retrieved_chunks'] if 'score' in c]
        if chunk_scores:
            scores.append(np.mean(chunk_scores))
    return np.mean(scores) if scores else 0

def avg_latency(results):
    return np.mean([r['latency'] for r in results])

def avg_answer_length(results):
    return np.mean([len(r['answer'].split()) for r in results])

# Display results table
comparison_data = {
    'Metric': [
        'Embedding Model', 'Vector DB', 'LLM', 'Top-K',
        'Avg Retrieval Score', 'Avg Latency (s)', 'Avg Answer Length (words)',
        'Embedding Dimensions'
    ],
    'Config A': [
        'MiniLM-L6', 'FAISS', 'LLaMA-3.1-8B', '3',
        f'{avg_retrieval_score(results_a):.4f}',
        f'{avg_latency(results_a):.2f}',
        f'{avg_answer_length(results_a):.0f}',
        '384'
    ],
    'Config B': [
        'MPNet-base', 'ChromaDB', 'Gemma-2-9B', '4',
        f'{avg_retrieval_score(results_b):.4f}',
        f'{avg_latency(results_b):.2f}',
        f'{avg_answer_length(results_b):.0f}',
        '768'
    ]
}

df = pd.DataFrame(comparison_data)
print('\n' + '='*60)
print('COMPARATIVE EVALUATION RESULTS')
print('='*60)
print(df.to_string(index=False))

Running comparative evaluation on all test queries...
(This may take ~1-2 minutes due to API calls)

  Query 1/5: What are the essential elements required for a val...
  Query 2/5: What are the consumer rights and remedies availabl...
  Query 3/5: What cyber crimes are defined under the IT Act and...
  Query 4/5: What is the doctrine of frustration in contract la...
  Query 5/5: Explain the fundamental rights guaranteed by the I...

COMPARATIVE EVALUATION RESULTS
                   Metric     Config A   Config B
          Embedding Model    MiniLM-L6 MPNet-base
                Vector DB        FAISS   ChromaDB
                      LLM LLaMA-3.1-8B Gemma-2-9B
                    Top-K            3          4
      Avg Retrieval Score       0.5990     0.2513
          Avg Latency (s)         0.78       0.93
Avg Answer Length (words)          268        246
     Embedding Dimensions          384        768


Step 12: Interactive Legal Assistant

In [ ]:

# Interactive mode — ask your own questions!

def ask_legal_question(question, config=CONFIG_A):
    """Ask any legal question and get an AI-powered answer."""
    print(f'❓ Question: {question}')
    print(f'⚙️  Using: {config["name"]}')
    print('\n🔍 Retrieving relevant legal texts...')

    result = rag_pipeline(question, config)

    print(f'\n📚 Retrieved Sources:')
    for i, chunk in enumerate(result['retrieved_chunks'], 1):
        print(f'  {i}. {chunk["metadata"]["source"][:55]} (score: {chunk["score"]:.4f})')

    print(f'\n⚖️  Legal Answer:')
    print('='*60)
    print(result['answer'])
    print('='*60)
    print(f'⏱️  Response time: {result["latency"]:.2f} seconds')
    return result

# Try your own question:
my_question = 'Can a minor enter into a valid contract in India?'
result = ask_legal_question(my_question)

❓ Question: Can a minor enter into a valid contract in India?
⚙️  Using: Config A: MiniLM + FAISS + LLaMA-3

🔍 Retrieving relevant legal texts...

📚 Retrieved Sources:
  1. Indian Contract Act, 1872 — Formation of Contract (score: 0.6473)
  2. Indian Contract Act, 1872 — Formation of Contract (score: 0.4811)
  3. Indian Contract Act, 1872 — Formation of Contract (score: 0.4794)

⚖️  Legal Answer:
**Answer:**

Based on the provided legal context, a minor cannot enter into a valid contract in India.

**Reasoning:**

As per the Indian Contract Act, 1872, a minor's contract is void ab initio. This is explicitly stated in the context as follows:

"[Source 1: Indian Contract Act, 1872 — Formation of Contract]
disqualified by law. A minor's contract is void ab initio."

This implies that a minor's contract is not only void but also void from the beginning. This means that a minor's contract is not enforceable by law and is not considered a valid contract.

**Conclusion:**

In India, a minor c